# ETL — Football Data API → Database

Pipeline completo: Extract → Transform → Load-ready DataFrames.

Fonte: [football-data.org](https://www.football-data.org/) (v4)  
Destino: PostgreSQL

In [ ]:
import json
import os
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

# --- Config ---
API_KEY = os.environ.get("FOOTBALL_DATA_API_TOKEN")
BASE_URL = os.environ.get("FOOTBALL_DATA_BASE_URL", "https://api.football-data.org/v4")
COMPETITION = os.environ.get("FOOTBALL_DATA_COMPETITION", "WC")
HEADERS = {"X-Auth-Token": API_KEY}

# Paths for raw and processed data
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


def api_get(endpoint: str) -> dict:
    """Call football-data.org API."""
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def save_raw(data: dict | list, filename: str) -> Path:
    """Save raw JSON to data/raw/."""
    path = DATA_RAW / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path} ({path.stat().st_size / 1024:.1f} KB)")
    return path


print(f"API Key: {'OK' if API_KEY else 'MISSING'}")
print(f"Competition: {COMPETITION}")

API Key: OK
Competition: WC


---
# EXTRACT

Puxar dados raw da API e guardar em `data/raw/` como JSON.

## E1. Competition

In [2]:
raw_competition = api_get(f"competitions/{COMPETITION}")
save_raw(raw_competition, "competition.json")

print(f"Season: {raw_competition['currentSeason']['startDate']} → {raw_competition['currentSeason']['endDate']}")
print(f"Season ID: {raw_competition['currentSeason']['id']}")

Saved: ../data/raw/competition.json (12.8 KB)
Season: 2026-06-11 → 2026-07-19
Season ID: 2398


## E2. Teams

In [3]:
raw_teams = api_get(f"competitions/{COMPETITION}/teams")
save_raw(raw_teams, "teams.json")

print(f"Total teams: {raw_teams['count']}")
for t in raw_teams["teams"][:5]:
    print(f"  {t['tla']} | {t['name']}")

Saved: ../data/raw/teams.json (283.5 KB)
Total teams: 48
  URY | Uruguay
  GER | Germany
  ESP | Spain
  PAR | Paraguay
  ARG | Argentina


## E3. Standings (Groups)

In [4]:
try:
    raw_standings = api_get(f"competitions/{COMPETITION}/standings")
    save_raw(raw_standings, "standings.json")
    print(f"Groups found: {len(raw_standings['standings'])}")
    for g in raw_standings["standings"]:
        teams_in_group = [e["team"]["tla"] for e in g["table"]]
        print(f"  {g['group']}: {', '.join(teams_in_group)}")
except Exception as e:
    print(f"Standings not available: {e}")
    raw_standings = None

Saved: ../data/raw/standings.json (24.3 KB)
Groups found: 12
  Group A: CZE, MEX, RSA, KOR
  Group B: BIH, CAN, QAT, SUI
  Group C: BRA, MAR, HAI, SCO
  Group D: TUR, USA, PAR, AUS
  Group E: GER, CUW, CIV, ECU
  Group F: SWE, NED, JPN, TUN
  Group G: BEL, EGY, IRN, NZL
  Group H: ESP, CPV, KSA, URY
  Group I: IRQ, FRA, SEN, NOR
  Group J: ARG, ALG, AUT, JOR
  Group K: COD, POR, UZB, COL
  Group L: ENG, CRO, GHA, PAN


## E4. Matches

In [ ]:
raw_matches = api_get(f"competitions/{COMPETITION}/matches")
save_raw(raw_matches, "matches.json")

print(f"Total matches: {raw_matches['resultSet']['count']}")

stages = Counter(m["stage"] for m in raw_matches["matches"])
for stage, count in stages.most_common():
    print(f"  {stage}: {count} matches")

Saved: ../data/raw/matches.json (142.9 KB)
Total matches: 104
  GROUP_STAGE: 72 matches
  LAST_32: 16 matches
  LAST_16: 8 matches
  QUARTER_FINALS: 4 matches
  SEMI_FINALS: 2 matches
  THIRD_PLACE: 1 matches
  FINAL: 1 matches


## E5. Squads (all teams)

Um request por equipa — com rate limiting (free tier: 10 req/min).

In [ ]:
import time

raw_squads = {}
team_ids = [(t["id"], t["tla"]) for t in raw_teams["teams"]]

print(f"Fetching squads for {len(team_ids)} teams (rate limited ~7s each)...")
print(f"Estimated time: ~{len(team_ids) * 7 / 60:.0f} min\n")

for i, (tid, tla) in enumerate(team_ids):
    if i > 0:
        time.sleep(7)

    team_detail = api_get(f"teams/{tid}")
    squad = team_detail.get("squad", [])
    raw_squads[tla] = {
        "team_id": tid,
        "team_tla": tla,
        "team_name": team_detail["name"],
        "squad": squad,
    }
    print(f"  [{i + 1}/{len(team_ids)}] {tla}: {len(squad)} players")

save_raw(raw_squads, "squads.json")
total_players = sum(len(s["squad"]) for s in raw_squads.values())
print(f"\nTotal players: {total_players}")

Fetching squads for 48 teams (rate limited ~7s each)...
Estimated time: ~6 min

  [1/48] URY: 26 players
  [2/48] GER: 26 players
  [3/48] ESP: 26 players
  [4/48] PAR: 26 players
  [5/48] ARG: 26 players
  [6/48] GHA: 26 players
  [7/48] BRA: 26 players


HTTPError: 429 Client Error:  for url: https://api.football-data.org/v4/teams/765

---
# TRANSFORM

Mapear os dados raw para o nosso schema de base de dados.

### Mappings de referência

```
API stage        → DB match_stage
GROUP_STAGE      → group
ROUND_OF_32      → R32
LAST_16          → R16
QUARTER_FINALS   → QF
SEMI_FINALS      → SF
THIRD_PLACE      → 3rd
FINAL            → F

API status       → DB match_status
TIMED            → scheduled
SCHEDULED        → scheduled
IN_PLAY          → live
PAUSED           → live
FINISHED         → finished
POSTPONED        → scheduled

API position     → DB player_position
Goalkeeper       → GK
Defence          → DF
Midfield         → MF
Offence          → FW
```

In [ ]:
# --- Mapping dictionaries ---
STAGE_MAP = {
    "GROUP_STAGE": "group",
    "ROUND_OF_32": "R32",
    "LAST_32": "R32",
    "LAST_16": "R16",
    "QUARTER_FINALS": "QF",
    "SEMI_FINALS": "SF",
    "THIRD_PLACE": "3rd",
    "FINAL": "F",
}

STATUS_MAP = {
    "TIMED": "scheduled",
    "SCHEDULED": "scheduled",
    "IN_PLAY": "live",
    "PAUSED": "live",
    "FINISHED": "finished",
    "POSTPONED": "scheduled",
    "CANCELLED": "scheduled",
    "SUSPENDED": "live",
}

POSITION_MAP = {
    "Goalkeeper": "GK",
    "Defence": "DF",
    "Midfield": "MF",
    "Offence": "FW",
}

DEADLINE_OFFSET = timedelta(hours=1)

print("Mappings loaded.")

Mappings loaded.


## T1. Transform Teams

API → `teams` table

In [ ]:
team_group_map = {}
if raw_standings:
    for group in raw_standings["standings"]:
        raw_group = group["group"]
        letter = raw_group.split(" ")[-1] if " " in raw_group else raw_group.replace("GROUP_", "")
        if len(letter) > 1:
            letter = letter[-1]
        for entry in group["table"]:
            tla = entry["team"]["tla"]
            team_group_map[tla] = letter

print(f"Group mapping: {len(team_group_map)} teams")
# Quick check
for tla, letter in list(team_group_map.items())[:6]:
    print(f"  {tla} → Group {letter}")

Group mapping: 48 teams
  CZE → Group A
  MEX → Group A
  RSA → Group A
  KOR → Group A
  BIH → Group B
  CAN → Group B


In [ ]:
# Region → Confederation (only 6 entries + 1 override)
# Built dynamically from the areas API endpoint
REGION_TO_CONFEDERATION = {
    "Europe": "UEFA",
    "South America": "CONMEBOL",
    "Africa": "CAF",
    "Asia": "AFC",
    "N/C America": "CONCACAF",
    "Oceania": "OFC",
}

CONFEDERATION_OVERRIDES = {
    "Australia": "AFC",  # geographically Oceania, plays in AFC
}

areas_path = DATA_RAW / "areas.json"
if areas_path.exists():
    raw_areas = json.load(open(areas_path))
else:
    raw_areas = api_get("areas")
    save_raw(raw_areas, "areas.json")

confed_lookup = {}
for area in raw_areas["areas"]:
    country = area["name"]
    parent = area.get("parentArea")
    if country in CONFEDERATION_OVERRIDES:
        confed_lookup[country] = CONFEDERATION_OVERRIDES[country]
    elif parent in REGION_TO_CONFEDERATION:
        confed_lookup[country] = REGION_TO_CONFEDERATION[parent]


teams_transformed = []
for t in raw_teams["teams"]:
    tla = t["tla"]
    area_name = t.get("area", {}).get("name", "")

    teams_transformed.append(
        {
            "name": t["name"],
            "code": tla,
            "flag_url": t.get("crest"),
            "group_letter": team_group_map.get(tla),
            "confederation": confed_lookup.get(area_name),
        }
    )

df_teams = pd.DataFrame(teams_transformed)
print(f"Teams: {len(df_teams)}")
print(f"With group: {df_teams['group_letter'].notna().sum()}")
print(f"With confederation: {df_teams['confederation'].notna().sum()}")
print()
df_teams.head(10)

Teams: 48
With group: 48
With confederation: 48



,name,code,flag_url,group_letter,confederation,_api_id,_area
0,Uruguay,URY,https://crests.football-data.org/758.svg,H,CONMEBOL,758,Uruguay
1,Germany,GER,https://crests.football-data.org/759.svg,E,UEFA,759,Germany
2,Spain,ESP,https://crests.football-data.org/760.svg,H,UEFA,760,Spain
3,Paraguay,PAR,https://crests.football-data.org/761.svg,D,CONMEBOL,761,Paraguay
4,Argentina,ARG,https://crests.football-data.org/762.png,J,CONMEBOL,762,Argentina
5,Ghana,GHA,https://crests.football-data.org/ghana.svg,L,CAF,763,Ghana
6,Brazil,BRA,https://crests.football-data.org/764.svg,C,CONMEBOL,764,Brazil
7,Portugal,POR,https://crests.football-data.org/765.svg,K,UEFA,765,Portugal
8,Japan,JPN,https://crests.football-data.org/766.svg,F,AFC,766,Japan
9,Mexico,MEX,https://crests.football-data.org/769.svg,A,CONCACAF,769,Mexico


In [ ]:
missing_confed = df_teams[df_teams["confederation"].isna()]
if len(missing_confed) > 0:
    print("Teams without confederation (need manual fix):")
    for _, row in missing_confed.iterrows():
        print(f"  {row['code']} | {row['name']} | area: {row['_area']}")
else:
    print("All teams have confederations!")

All teams have confederations!


## T2. Transform Matches

API → `matches` table

In [ ]:
tla_to_api_id = {t["tla"]: t["id"] for t in raw_teams["teams"]}
api_id_to_tla = {v: k for k, v in tla_to_api_id.items()}


def extract_group_letter(group_str: str | None) -> str | None:
    """'GROUP_A' → 'A', None → None."""
    if not group_str:
        return None
    cleaned = group_str.split(" ")[-1] if " " in group_str else group_str.replace("GROUP_", "")
    return cleaned[-1] if len(cleaned) > 1 else cleaned


def make_placeholder(stage: str, match: dict) -> tuple[str | None, str | None]:
    """Generate placeholders for knockout matches without teams yet."""
    home_id = match.get("homeTeam", {}).get("id")
    away_id = match.get("awayTeam", {}).get("id")

    home_ph = None if home_id else "W?"  # e.g. winner of match X
    away_ph = None if away_id else "W?"
    return home_ph, away_ph


matches_transformed = []
for i, m in enumerate(raw_matches["matches"], start=1):
    stage_api = m["stage"]
    stage_db = STAGE_MAP.get(stage_api)
    if not stage_db:
        print(f"  WARNING: unknown stage '{stage_api}' in match {m['id']}")
        continue

    status_db = STATUS_MAP.get(m["status"], "scheduled")

    home_api_id = m.get("homeTeam", {}).get("id")
    away_api_id = m.get("awayTeam", {}).get("id")
    home_tla = api_id_to_tla.get(home_api_id) if home_api_id else None
    away_tla = api_id_to_tla.get(away_api_id) if away_api_id else None

    home_ph = None if home_tla else (m.get("homeTeam", {}).get("name") or "TBD")
    away_ph = None if away_tla else (m.get("awayTeam", {}).get("name") or "TBD")

    ft = m.get("score", {}).get("fullTime", {})
    home_score = ft.get("home") if status_db == "finished" else None
    away_score = ft.get("away") if status_db == "finished" else None

    match_date = m["utcDate"]
    match_dt = datetime.fromisoformat(match_date.replace("Z", "+00:00"))
    deadline_dt = match_dt - DEADLINE_OFFSET

    matches_transformed.append(
        {
            "match_number": i,
            "stage": stage_db,
            "group_letter": extract_group_letter(m.get("group")),
            "home_team_code": home_tla,
            "away_team_code": away_tla,
            "home_placeholder": home_ph,
            "away_placeholder": away_ph,
            "match_date": match_date,
            "submission_deadline": deadline_dt.isoformat(),
            "venue": m.get("venue"),
            "status": status_db,
            "home_score": home_score,
            "away_score": away_score,
            "_api_id": m["id"],
            "_api_matchday": m["matchday"],
            "_api_status": m["status"],
        }
    )

df_matches = pd.DataFrame(matches_transformed)
print(f"Matches: {len(df_matches)}")
print("\nBy stage:")
print(df_matches["stage"].value_counts().to_string())
print(
    f"\nWith both teams defined: {(df_matches['home_team_code'].notna() & df_matches['away_team_code'].notna()).sum()}"
)
print(f"Knockout TBD: {(df_matches['home_placeholder'].notna() | df_matches['away_placeholder'].notna()).sum()}")

Matches: 104

By stage:
stage
group    72
R32      16
R16       8
QF        4
SF        2
3rd       1
F         1

With both teams defined: 72
Knockout TBD: 32


In [12]:
# Preview group stage matches
df_group = df_matches[df_matches["stage"] == "group"].copy()
print(f"Group stage: {len(df_group)} matches")
df_group[["match_number", "group_letter", "home_team_code", "away_team_code", "match_date", "venue"]].head(12)

Group stage: 72 matches


,match_number,group_letter,home_team_code,away_team_code,match_date,venue
0,1,A,MEX,RSA,2026-06-11T19:00:00Z,None
1,2,A,KOR,CZE,2026-06-12T02:00:00Z,None
2,3,B,CAN,BIH,2026-06-12T19:00:00Z,None
3,4,D,USA,PAR,2026-06-13T01:00:00Z,None
4,5,B,QAT,SUI,2026-06-13T19:00:00Z,None
5,6,C,BRA,MAR,2026-06-13T22:00:00Z,None
6,7,C,HAI,SCO,2026-06-14T01:00:00Z,None
7,8,D,AUS,TUR,2026-06-14T04:00:00Z,None
8,9,E,GER,CUW,2026-06-14T17:00:00Z,None
9,10,F,NED,JPN,2026-06-14T20:00:00Z,None


In [13]:
# Preview knockout matches (check placeholders)
df_knockout = df_matches[df_matches["stage"] != "group"].copy()
print(f"Knockout: {len(df_knockout)} matches")
df_knockout[["match_number", "stage", "home_team_code", "home_placeholder", "away_team_code", "away_placeholder"]].head(
    10
)

Knockout: 32 matches


,match_number,stage,home_team_code,home_placeholder,away_team_code,away_placeholder
72,73,R32,NaN,TBD,NaN,TBD
73,74,R32,NaN,TBD,NaN,TBD
74,75,R32,NaN,TBD,NaN,TBD
75,76,R32,NaN,TBD,NaN,TBD
76,77,R32,NaN,TBD,NaN,TBD
77,78,R32,NaN,TBD,NaN,TBD
78,79,R32,NaN,TBD,NaN,TBD
79,80,R32,NaN,TBD,NaN,TBD
80,81,R32,NaN,TBD,NaN,TBD
81,82,R32,NaN,TBD,NaN,TBD


## T3. Transform Players

API → `players` table

In [14]:
players_transformed = []
skipped = []

for tla, team_data in raw_squads.items():
    for p in team_data["squad"]:
        position_api = p.get("position")
        position_db = POSITION_MAP.get(position_api)

        if not position_db:
            skipped.append((tla, p["name"], position_api))
            continue

        players_transformed.append(
            {
                "team_code": tla,
                "name": p["name"],
                "position": position_db,
                "birth_date": p.get("dateOfBirth"),
                # Metadata
                "_api_id": p["id"],
                "_nationality": p.get("nationality"),
            }
        )

df_players = pd.DataFrame(players_transformed)
print(f"Players: {len(df_players)}")
print(f"Skipped (no valid position): {len(skipped)}")
print("\nBy position:")
print(df_players["position"].value_counts().to_string())

if skipped:
    print("\nSkipped players (first 10):")
    for tla, name, pos in skipped[:10]:
        print(f"  {tla} | {name} | position: {pos}")

Players: 182
Skipped (no valid position): 0

By position:
position
DF    58
MF    57
FW    46
GK    21


In [15]:
# Preview
df_players.head(10)

,team_code,name,position,birth_date,_api_id,_nationality
0,URY,Fernando Muslera,GK,1986-06-16,3160,Uruguay
1,URY,Santiago Mele,GK,1997-09-06,30210,Uruguay
2,URY,Sergio Rochet,GK,1993-03-23,30386,Uruguay
3,URY,Guillermo Varela,DF,1993-03-24,3165,Uruguay
4,URY,Ronald Araújo,DF,1999-03-07,28292,Uruguay
5,URY,Sebastían Cáceres,DF,1999-08-18,28326,Uruguay
6,URY,Matías Viña,DF,1997-11-09,28426,Uruguay
7,URY,Joaquín Piquerez,DF,1998-08-24,28740,Uruguay
8,URY,Maximiliano Araújo,DF,2000-02-15,28770,Uruguay
9,URY,Mathías Olivera,DF,1997-10-31,32754,Uruguay


---
# VALIDATION

Verificar consistência antes de avançar para o Load.

In [16]:
errors = []

# 1. All team codes are unique and 3 chars
dupes = df_teams[df_teams["code"].duplicated()]
if len(dupes) > 0:
    errors.append(f"Duplicate team codes: {dupes['code'].tolist()}")

bad_codes = df_teams[df_teams["code"].str.len() != 3]
if len(bad_codes) > 0:
    errors.append(f"Invalid code length: {bad_codes['code'].tolist()}")

# 2. All match team_codes exist in teams
valid_codes = set(df_teams["code"])
for col in ["home_team_code", "away_team_code"]:
    match_codes = set(df_matches[col].dropna())
    unknown = match_codes - valid_codes
    if unknown:
        errors.append(f"Unknown {col}: {unknown}")

# 3. All player team_codes exist in teams
player_codes = set(df_players["team_code"])
unknown_player_teams = player_codes - valid_codes
if unknown_player_teams:
    errors.append(f"Unknown player teams: {unknown_player_teams}")

# 4. Group letters are single chars A-L
valid_groups = set("ABCDEFGHIJKL")
team_groups = set(df_teams["group_letter"].dropna())
bad_groups = team_groups - valid_groups
if bad_groups:
    errors.append(f"Invalid group letters: {bad_groups}")

# 5. Stages are valid enum values
valid_stages = {"group", "R32", "R16", "QF", "SF", "3rd", "F"}
match_stages = set(df_matches["stage"])
bad_stages = match_stages - valid_stages
if bad_stages:
    errors.append(f"Invalid stages: {bad_stages}")

# 6. Positions are valid enum values
valid_positions = {"GK", "DF", "MF", "FW"}
player_positions = set(df_players["position"])
bad_positions = player_positions - valid_positions
if bad_positions:
    errors.append(f"Invalid positions: {bad_positions}")

# 7. Match numbers are unique and sequential
if df_matches["match_number"].duplicated().any():
    errors.append("Duplicate match numbers")

# 8. Group matches have group_letter, knockout don't
group_no_letter = df_matches[(df_matches["stage"] == "group") & (df_matches["group_letter"].isna())]
if len(group_no_letter) > 0:
    errors.append(f"Group matches without group_letter: {len(group_no_letter)}")

# 9. submission_deadline < match_date (DB constraint)
df_check = df_matches.copy()
df_check["_md"] = pd.to_datetime(df_check["match_date"])
df_check["_sd"] = pd.to_datetime(df_check["submission_deadline"])
bad_deadlines = df_check[df_check["_sd"] >= df_check["_md"]]
if len(bad_deadlines) > 0:
    errors.append(f"Deadline >= match_date: {len(bad_deadlines)} matches")

# Results
if errors:
    print("VALIDATION ERRORS:")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("All validations passed!")

print("\nSummary:")
print(f"  Teams:   {len(df_teams)}")
print(f"  Matches: {len(df_matches)}")
print(f"  Players: {len(df_players)}")

All validations passed!

Summary:
  Teams:   48
  Matches: 104
  Players: 182


---
# SAVE PROCESSED DATA

Guardar os DataFrames transformados em `data/processed/` para uso nos scripts de Load.

In [17]:
# Save as CSV (easy to inspect) — drop _api columns
db_cols_teams = ["name", "code", "flag_url", "group_letter", "confederation"]
df_teams[db_cols_teams].to_csv(DATA_PROCESSED / "teams.csv", index=False)

db_cols_matches = [
    "match_number",
    "stage",
    "group_letter",
    "home_team_code",
    "away_team_code",
    "home_placeholder",
    "away_placeholder",
    "match_date",
    "submission_deadline",
    "venue",
    "status",
    "home_score",
    "away_score",
]
df_matches[db_cols_matches].to_csv(DATA_PROCESSED / "matches.csv", index=False)

db_cols_players = ["team_code", "name", "position", "birth_date"]
df_players[db_cols_players].to_csv(DATA_PROCESSED / "players.csv", index=False)

print("Saved to data/processed/:")
for f in sorted(DATA_PROCESSED.glob("*.csv")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Saved to data/processed/:
  matches.csv (8.3 KB)
  players.csv (5.9 KB)
  teams.csv (3.0 KB)
